# ML Testing & MLOps: Schema Contracts and Golden-Master Regression Testing for Retrained Models

**Dataset:** [Titanic passenger manifest](https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv)
— 891 real passenger records from the RMS Titanic (class, sex, age, family size, fare,
port of embarkation, and whether they survived), loaded directly from the `seaborn-data`
CSV mirror with `pandas.read_csv`. It has genuine messiness (177 missing ages, 2 missing
embarkation ports) which is exactly what makes it useful for data-validation testing.

## Problem statement

You are the ML quality engineer for a Titanic survival-prediction service. The model gets
retrained periodically as the team tweaks the feature pipeline. Your job is to build the two
guardrails that stop a bad retrain from ever reaching production — the same instincts you'd
bring from SQA regression testing, applied to a model instead of a binary:

1. **A data contract test** that any dataframe must pass *before* anything is trained on it:
   right columns, right dtypes, nulls only where expected (and not too many), categorical
   columns containing only allowed values, numeric columns inside sane physical ranges.
2. **A golden-master regression gate** that compares a freshly retrained *candidate* model
   against the currently-deployed *baseline* model on a frozen validation set, and blocks
   promotion if either:
   - the candidate's **recall** on survivors drops by more than a small tolerance versus the
     baseline (a metric regression — not just "did accuracy go up"), or
   - too many individual rows get a **different prediction** between candidate and baseline
     (prediction-level divergence — the golden-master idea ported from SQA snapshot testing).

## What to produce

Concretely, by the end of this notebook you should have:

- `DataContract` — a class holding an expected schema (columns/dtypes, allowed null rates,
  allowed categorical values, numeric ranges) with a `.validate(df)` method returning
  `{"passed": bool, "violations": [...]}`.
- `evaluate(pipeline, X, y)` — returns a metrics dict (`accuracy`, `precision`, `recall`, `f1`)
  plus the raw predictions.
- `regression_gate(baseline_metrics, candidate_metrics, metric="recall", tolerance=0.03)` —
  returns a pass/fail report on whether the candidate regressed beyond tolerance.
- `prediction_divergence(y_pred_baseline, y_pred_candidate, max_divergence_rate=0.10)` — a
  row-level golden-master diff: fraction of rows where the two models disagree, plus which
  rows, and a pass/fail against a max divergence rate.
- A final `promote = ...` decision that only promotes the candidate if **both** gates pass,
  run against a realistic scenario below where a retrain accidentally ships without the `sex`
  feature (a real class of MLOps bug: a feature-pipeline refactor silently drops a column).

---

## Starter cells

Fill in every `# TODO`. Don't look at the solution until you've tried each step.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

RANDOM_STATE = 42

df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")

FEATURES = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
NUMERIC = ["pclass", "age", "sibsp", "parch", "fare"]
CATEGORICAL = ["sex", "embarked"]

df[FEATURES + ["survived"]].head()

In [ ]:
# TODO: define EXPECTED_SCHEMA, a dict describing the contract this data must satisfy:
#   - "columns": {col_name: logical_kind} for every column in FEATURES + ["survived"],
#       where logical_kind is one of "integer", "float", "string"
#       (survived, pclass, sibsp, parch -> "integer"; age, fare -> "float"; sex, embarked -> "string")
#   - "nullable": {col_name: max_null_rate} for the only two columns allowed to have nulls
#       (age can be up to 25% null, embarked up to 1% null). Every other column must have zero nulls.
#   - "categorical_values": {col_name: set_of_allowed_values} for sex, embarked, and pclass
#   - "numeric_ranges": {col_name: (low, high)} for age, fare, sibsp, parch — pick sane physical bounds

EXPECTED_SCHEMA = {
    "columns": {},  # TODO
    "nullable": {},  # TODO
    "categorical_values": {},  # TODO
    "numeric_ranges": {},  # TODO
}

In [ ]:
# TODO: implement DataContract.validate(df). It should check, against self.schema:
#   1. every expected column is present, and its dtype matches its logical kind
#      (hint: use pd.api.types.is_integer_dtype / is_float_dtype, and treat a column as
#       "string"-like if pd.api.types.is_string_dtype(s) or pd.api.types.is_object_dtype(s) —
#       don't compare dtype strings directly, pandas versions disagree on what a string
#       column's dtype prints as)
#   2. every column in "nullable" has a null rate <= its allowed max
#   3. every OTHER column (in "columns" but not in "nullable") has zero nulls
#   4. every column in "categorical_values" contains only values from its allowed set
#      (ignore NaN when checking this)
#   5. every column in "numeric_ranges" has all non-null values inside [low, high]
# Collect every violation as a short human-readable string. Return
#   {"passed": len(violations) == 0, "violations": violations}

class DataContract:
    def __init__(self, schema):
        self.schema = schema

    def validate(self, df):
        violations = []
        # TODO
        return {"passed": len(violations) == 0, "violations": violations}


contract = DataContract(EXPECTED_SCHEMA)
report = contract.validate(df)
print(report)
assert report["passed"], "the raw titanic data should satisfy its own contract"

In [ ]:
# TODO: split df into X_train, X_val, y_train, y_val using train_test_split with
# test_size=0.25, stratify=y, random_state=RANDOM_STATE, where X = df[FEATURES] and
# y = df["survived"].
#
# Then write build_pipeline(feature_cols, numeric_cols, categorical_cols) that returns a
# sklearn Pipeline: a ColumnTransformer with
#   - numeric branch: SimpleImputer(strategy="median") -> StandardScaler(), applied to
#     whichever of numeric_cols are in feature_cols
#   - categorical branch: SimpleImputer(strategy="most_frequent") -> OneHotEncoder(handle_unknown="ignore"),
#     applied to whichever of categorical_cols are in feature_cols
# feeding into a LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).

X_train, X_val, y_train, y_val = ...  # TODO

def build_pipeline(feature_cols, numeric_cols, categorical_cols):
    ...  # TODO


# Simulate the real MLOps scenario: "baseline" is the currently-deployed model, trained on
# the full FEATURES set. "candidate" is a fresh retrain where a feature-pipeline refactor
# accidentally dropped "sex" before it reached the model.
baseline_features = FEATURES
candidate_features = ["pclass", "age", "sibsp", "parch", "fare", "embarked"]

# TODO: build and fit baseline_pipeline on X_train[baseline_features], y_train
# TODO: build and fit candidate_pipeline on X_train[candidate_features], y_train
baseline_pipeline = ...  # TODO
candidate_pipeline = ...  # TODO

In [ ]:
# TODO: implement evaluate(pipeline, X_eval, y_eval) -> (metrics_dict, y_pred)
# metrics_dict should have keys "accuracy", "precision", "recall", "f1" from sklearn.metrics,
# computed on pipeline.predict(X_eval) vs y_eval.

def evaluate(pipeline, X_eval, y_eval):
    ...  # TODO


baseline_metrics, baseline_preds = evaluate(baseline_pipeline, X_val[baseline_features], y_val)
candidate_metrics, candidate_preds = evaluate(candidate_pipeline, X_val[candidate_features], y_val)
print("baseline:", baseline_metrics)
print("candidate:", candidate_metrics)

In [ ]:
# TODO: implement regression_gate(baseline_metrics, candidate_metrics, metric="recall", tolerance=0.03)
# -> dict with "passed" (bool: baseline[metric] - candidate[metric] <= tolerance), plus the
# baseline value, candidate value, and the drop, for reporting.

def regression_gate(baseline_metrics, candidate_metrics, metric="recall", tolerance=0.03):
    ...  # TODO


# TODO: implement prediction_divergence(y_pred_baseline, y_pred_candidate, max_divergence_rate=0.10)
# -> dict with "passed" (bool), "divergence_rate" (fraction of rows where the two prediction
# arrays differ), and "diverging_indices" (list of row indices where they differ, first 20).

def prediction_divergence(y_pred_baseline, y_pred_candidate, max_divergence_rate=0.10):
    ...  # TODO


gate_report = regression_gate(baseline_metrics, candidate_metrics)
div_report = prediction_divergence(baseline_preds, candidate_preds)
print("regression gate:", gate_report)
print("divergence report:", div_report)

# TODO: promote should be True only if BOTH gate_report["passed"] and div_report["passed"] are True
promote = ...  # TODO
print("PROMOTE CANDIDATE:", promote)

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

RANDOM_STATE = 42

df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")

FEATURES = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
NUMERIC = ["pclass", "age", "sibsp", "parch", "fare"]
CATEGORICAL = ["sex", "embarked"]

EXPECTED_SCHEMA = {
    "columns": {
        "survived": "integer", "pclass": "integer", "sex": "string",
        "age": "float", "sibsp": "integer", "parch": "integer",
        "fare": "float", "embarked": "string",
    },
    "nullable": {"age": 0.25, "embarked": 0.01},
    "categorical_values": {
        "sex": {"male", "female"},
        "embarked": {"C", "Q", "S"},
        "pclass": {1, 2, 3},
    },
    "numeric_ranges": {"age": (0, 100), "fare": (0, 600), "sibsp": (0, 10), "parch": (0, 10)},
}

_DTYPE_CHECKS = {
    "integer": pd.api.types.is_integer_dtype,
    "float": pd.api.types.is_float_dtype,
    "string": lambda s: pd.api.types.is_string_dtype(s) or pd.api.types.is_object_dtype(s),
}

class DataContract:
    def __init__(self, schema):
        self.schema = schema

    def validate(self, df):
        violations = []

        for col, kind in self.schema["columns"].items():
            if col not in df.columns:
                violations.append(f"missing column: {col}")
                continue
            if not _DTYPE_CHECKS[kind](df[col]):
                violations.append(f"column {col} has dtype {df[col].dtype}, expected {kind}-like")

        for col, max_null_rate in self.schema["nullable"].items():
            if col in df.columns:
                null_rate = df[col].isnull().mean()
                if null_rate > max_null_rate:
                    violations.append(f"column {col} null rate {null_rate:.3f} exceeds max {max_null_rate}")

        non_nullable = set(self.schema["columns"]) - set(self.schema["nullable"])
        for col in non_nullable:
            if col in df.columns and df[col].isnull().any():
                violations.append(f"column {col} has unexpected nulls (not in nullable set)")

        for col, allowed in self.schema["categorical_values"].items():
            if col in df.columns:
                observed = set(df[col].dropna().unique())
                unexpected = observed - allowed
                if unexpected:
                    violations.append(f"column {col} has unexpected values: {unexpected}")

        for col, (lo, hi) in self.schema["numeric_ranges"].items():
            if col in df.columns:
                vals = df[col].dropna()
                bad = vals[(vals < lo) | (vals > hi)]
                if len(bad) > 0:
                    violations.append(f"column {col} has {len(bad)} values outside [{lo}, {hi}]")

        return {"passed": len(violations) == 0, "violations": violations}


contract = DataContract(EXPECTED_SCHEMA)
report = contract.validate(df)
print(report)
assert report["passed"]

X = df[FEATURES]
y = df["survived"]
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

def build_pipeline(feature_cols, numeric_cols, categorical_cols):
    numeric_cols = [c for c in numeric_cols if c in feature_cols]
    categorical_cols = [c for c in categorical_cols if c in feature_cols]
    pre = ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric_cols),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
    ])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])

baseline_features = FEATURES
candidate_features = ["pclass", "age", "sibsp", "parch", "fare", "embarked"]  # "sex" dropped by a pipeline bug

baseline_pipeline = build_pipeline(baseline_features, NUMERIC, CATEGORICAL)
baseline_pipeline.fit(X_train[baseline_features], y_train)

candidate_pipeline = build_pipeline(candidate_features, NUMERIC, CATEGORICAL)
candidate_pipeline.fit(X_train[candidate_features], y_train)

def evaluate(pipeline, X_eval, y_eval):
    y_pred = pipeline.predict(X_eval)
    return {
        "accuracy": accuracy_score(y_eval, y_pred),
        "precision": precision_score(y_eval, y_pred),
        "recall": recall_score(y_eval, y_pred),
        "f1": f1_score(y_eval, y_pred),
    }, y_pred

baseline_metrics, baseline_preds = evaluate(baseline_pipeline, X_val[baseline_features], y_val)
candidate_metrics, candidate_preds = evaluate(candidate_pipeline, X_val[candidate_features], y_val)
print("baseline:", baseline_metrics)
print("candidate:", candidate_metrics)

def regression_gate(baseline_metrics, candidate_metrics, metric="recall", tolerance=0.03):
    drop = baseline_metrics[metric] - candidate_metrics[metric]
    return {
        "passed": drop <= tolerance,
        "metric": metric,
        "baseline": baseline_metrics[metric],
        "candidate": candidate_metrics[metric],
        "drop": drop,
        "tolerance": tolerance,
    }

def prediction_divergence(y_pred_baseline, y_pred_candidate, max_divergence_rate=0.10):
    y_pred_baseline = np.asarray(y_pred_baseline)
    y_pred_candidate = np.asarray(y_pred_candidate)
    diverging = np.where(y_pred_baseline != y_pred_candidate)[0]
    rate = len(diverging) / len(y_pred_baseline)
    return {
        "passed": rate <= max_divergence_rate,
        "divergence_rate": rate,
        "diverging_indices": diverging.tolist()[:20],
    }

gate_report = regression_gate(baseline_metrics, candidate_metrics)
div_report = prediction_divergence(baseline_preds, candidate_preds)
print("regression gate:", gate_report)
print("divergence report:", div_report)

promote = gate_report["passed"] and div_report["passed"]
print("PROMOTE CANDIDATE:", promote)
```

**What this shows:** the data contract passes cleanly on the raw Titanic CSV — it's genuinely
messy (real nulls in `age`/`embarked`) but within the bounds we declared acceptable, which is
the point of a contract: it encodes *what messiness is tolerated*, not "zero nulls ever."

The retrain scenario is where it gets interesting. Dropping `sex` sounds like a small feature
pipeline change, but `sex` is by far the strongest predictor of survival on the Titanic ("women
and children first"). The candidate's **recall** on survivors collapses from roughly 0.69 to
0.43 — a huge, silent regression that `regression_gate` catches immediately (drop ≈ 0.26,
tolerance 0.03). Independently, `prediction_divergence` shows roughly 31% of validation rows
flip to a different prediction — way above a 10% golden-master tolerance. Either gate alone
would have blocked this retrain; together they catch both "the aggregate number got worse" and
"individual predictions silently changed," which is exactly the pairing of regression testing
(assert on behavior) and golden-master diffing (assert on outputs matching a known-good
reference) that SQA already does for non-ML software — just pointed at a model.

</details>